In [ ]:
import os, glob, math, random, shutil, subprocess, json, base64, zlib, re, sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUN_TRAINING = True
RUN_INFERENCE_AFTER_TRAIN = True

MODEL_VARIANT = "full_hat"
SCALE = 4

BATCH_SIZE = 2
GT_SIZE = 256

TOTAL_TARGET_ITER = 60000
ITERS_PER_SESSION = 10000
AUTO_RESUME_FROM_INPUT = True

TOTAL_ITER = None
RESUME_ITER = 0
AUTO_RESUME = False
RESUME_STATE_PATH = None

LEARNING_RATE = 2e-4
COSINE_ETA_MIN = 1e-5
COSINE_PERIODS = [TOTAL_TARGET_ITER]
COSINE_RESTART_WEIGHTS = [1.0]
USE_COSINE_LR = True
USE_CONSTANT_LR = False
WARMUP_ITER = -1

ADAM_BETAS = [0.9, 0.99]
WEIGHT_DECAY = 0.0
EMA_DECAY = 0.9995

LOSS_TYPE = "l1" 
CLEAN_PSNR_THRESHOLD = 19.0
USE_CLEAN_SYMLINK_DATA = True
USE_PREVIOUS_CLEAN_REPORT = True
PREVIOUS_CLEAN_REPORT = "/kaggle/input/username/clean_pair_report.csv"
FORCE_REBUILD_CLEAN_DATA = True

USE_HFLIP = True
USE_ROT = True

ENABLE_HOLDOUT_VAL = False
VALIDATE_DURING_TRAIN = False

SAVE_CHECKPOINT_FREQ = 1000
EXPORT_RESUME_PACKAGE = True
FORCE_DELETE_EXISTING_EXPERIMENT = True
AUTO_SKIP_IF_TARGET_REACHED = False

RUN_TTA_SUBMISSION = True
USE_TTA = True
TTA_MODES = list(range(8))
USE_HALF = False
ENCODE_BGR = True
WINDOW_SIZE = 16

WORK_DIR = "/kaggle/working"
HAT_DIR = f"{WORK_DIR}/HAT"
MODEL_DIR = f"{WORK_DIR}/downloaded_models"

BASE_EXP_NAME = "vg_full_hat_restart_clean20_b2_cosine_l1_30k_ckpt"
EXP_NAME = f"{BASE_EXP_NAME}_{MODEL_VARIANT}"

CLEAN_HR = f"{WORK_DIR}/clean_train/hr"
CLEAN_LR = f"{WORK_DIR}/clean_train/lr"
REPORT_CSV = f"{WORK_DIR}/clean_pair_report.csv"
CLEAN_CONFIG_JSON = f"{WORK_DIR}/clean_config.json"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

COMP_CANDIDATES = [
    "/kaggle/input/competitions/super-resolution-in-video-games",
    "/kaggle/input/super-resolution-in-video-games",
]
COMP_DIR = None
for p in COMP_CANDIDATES:
    if (os.path.isdir(f"{p}/train/hr") and os.path.isdir(f"{p}/train/lr") and
        os.path.isdir(f"{p}/test/lr") and os.path.isfile(f"{p}/sample_submission.csv")):
        COMP_DIR = p
        break

if COMP_DIR is None:
    for sub_path in glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True):
        root = os.path.dirname(sub_path)
        if os.path.isdir(f"{root}/train/hr") and os.path.isdir(f"{root}/train/lr") and os.path.isdir(f"{root}/test/lr"):
            COMP_DIR = root
            break

if COMP_DIR is None:
    raise FileNotFoundError("Cannot find competition data. Add the competition dataset to Kaggle input.")

TRAIN_HR = f"{COMP_DIR}/train/hr"
TRAIN_LR = f"{COMP_DIR}/train/lr"
TEST_LR = f"{COMP_DIR}/test/lr"
SAMPLE_SUB = f"{COMP_DIR}/sample_submission.csv"

print("Configuration loaded.")
print("COMP_DIR                 =", COMP_DIR)
print("EXP_NAME                 =", EXP_NAME)
print("MODEL_VARIANT            =", MODEL_VARIANT)
print("BATCH_SIZE               =", BATCH_SIZE)
print("GT_SIZE                  =", GT_SIZE)
print("TOTAL_TARGET_ITER        =", TOTAL_TARGET_ITER)
print("ITERS_PER_SESSION        =", ITERS_PER_SESSION)
print("LEARNING_RATE            =", LEARNING_RATE)
print("COSINE_ETA_MIN           =", COSINE_ETA_MIN)
print("LOSS_TYPE                =", LOSS_TYPE)
print("CLEAN_PSNR_THRESHOLD     =", CLEAN_PSNR_THRESHOLD)
print("EMA_DECAY                =", EMA_DECAY)
print("AUTO_RESUME_FROM_INPUT   =", AUTO_RESUME_FROM_INPUT)
print("RUN_TTA_SUBMISSION       =", RUN_TTA_SUBMISSION)


In [ ]:
def run(cmd, cwd=None, check=True):
    print("\n$ " + cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

run("pip install -q pyyaml opencv-python pillow pandas tqdm numpy scipy scikit-image einops timm requests", check=False)

if os.path.exists(HAT_DIR):
    print("HAT repo already exists:", HAT_DIR)
else:
    run(f"git clone --depth 1 https://github.com/XPixelGroup/HAT.git {HAT_DIR}")

run("pip install -q -r requirements.txt", cwd=HAT_DIR, check=False)
run("python setup.py develop", cwd=HAT_DIR, check=False)

if HAT_DIR not in sys.path:
    sys.path.insert(0, HAT_DIR)

candidate_paths = glob.glob("/usr/local/lib/python*/dist-packages/basicsr/data/degradations.py")
candidate_paths += glob.glob("/opt/conda/lib/python*/site-packages/basicsr/data/degradations.py")
for p in candidate_paths:
    if os.path.exists(p):
        txt = Path(p).read_text(encoding="utf-8")
        old = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
        new = "from torchvision.transforms.functional import rgb_to_grayscale"
        if old in txt:
            Path(p).write_text(txt.replace(old, new), encoding="utf-8")
            print("Patched BasicSR:", p)

try:
    import torchvision.transforms
    transforms_dir = os.path.dirname(torchvision.transforms.__file__)
    shim_path = os.path.join(transforms_dir, "functional_tensor.py")
    if not os.path.exists(shim_path):
        Path(shim_path).write_text("from .functional import rgb_to_grayscale\n", encoding="utf-8")
        print("Created shim:", shim_path)
except Exception as e:
    print("Torchvision shim warning:", repr(e))

def find_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    matches += glob.glob(f"/kaggle/working/**/{filename}", recursive=True)
    return sorted(set([m for m in matches if os.path.isfile(m)]))

target_name = "HAT_SRx4.pth"
matches = find_file(target_name)
if matches:
    MODEL_PATH = matches[0]
    print("Found full HAT checkpoint:", MODEL_PATH)
else:
    MODEL_PATH = f"{MODEL_DIR}/HAT_SRx4.pth"
    url = "https://huggingface.co/jaideepsingh/upscale_models/resolve/main/HAT/HAT_SRx4.pth?download=true"
    print("Trying download:", url)
    subprocess.run(f'wget -q --show-progress -O "{MODEL_PATH}" "{url}"', shell=True)
    if not (os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 50 * 1024 * 1024):
        raise FileNotFoundError("Cannot find/download HAT_SRx4.pth. Enable Kaggle Internet or upload HAT_SRx4.pth as input.")

PRETRAIN_PATH = MODEL_PATH
PARAM_KEY_G = "params_ema"
STRICT_LOAD_G = True

print("Checkpoint ready.")
print("MODEL_PATH =", MODEL_PATH)
print("Size MB    =", round(os.path.getsize(MODEL_PATH) / 1024 / 1024, 2))


In [ ]:
import cv2

def imread_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def calc_psnr(img1, img2):
    img1 = img1.astype(np.float64)
    img2 = img2.astype(np.float64)
    mse = np.mean((img1 - img2) ** 2)
    if mse <= 1e-12:
        return 99.0
    return 20 * math.log10(255.0 / math.sqrt(mse))

def safe_link_or_copy(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)

def find_name_col(df):
    for c in df.columns:
        lc = c.lower()
        if "filename" in lc or lc in ["file", "name", "image", "path"]:
            return c
    return df.columns[0]

def find_psnr_col(df):
    for c in df.columns:
        if "psnr" in c.lower():
            return c
    return None

hr_files = {os.path.basename(p): p for p in glob.glob(os.path.join(TRAIN_HR, "*"))}
lr_files = {os.path.basename(p): p for p in glob.glob(os.path.join(TRAIN_LR, "*"))}
common = sorted(set(hr_files) & set(lr_files))
print("Common train pairs:", len(common))

report_loaded = False
if USE_PREVIOUS_CLEAN_REPORT and os.path.exists(PREVIOUS_CLEAN_REPORT):
    print("Using previous clean report:", PREVIOUS_CLEAN_REPORT)
    rep = pd.read_csv(PREVIOUS_CLEAN_REPORT)
    name_col = find_name_col(rep)
    psnr_col = find_psnr_col(rep)
    if psnr_col is None:
        raise ValueError(f"Cannot find PSNR column in report. Columns={rep.columns.tolist()}")
    rep = rep.rename(columns={name_col: "filename", psnr_col: "bicubic_psnr"})
    report_loaded = True
else:
    print("No valid previous report. Computing bicubic PSNR report.")

if not report_loaded:
    rows = []
    for fn in tqdm(common, desc="Checking train pairs"):
        hr = imread_rgb(hr_files[fn])
        lr = imread_rgb(lr_files[fn])
        up = cv2.resize(lr, (hr.shape[1], hr.shape[0]), interpolation=cv2.INTER_CUBIC)
        rows.append({"filename": fn, "bicubic_psnr": calc_psnr(up, hr)})
    rep = pd.DataFrame(rows)

rep["filename"] = rep["filename"].astype(str)
rep.to_csv(REPORT_CSV, index=False)

clean_names = sorted(set(rep.loc[rep["bicubic_psnr"] >= CLEAN_PSNR_THRESHOLD, "filename"]) & set(common))
print("Clean pairs:", len(clean_names))
print("Removed:", len(common) - len(clean_names))

if os.path.exists(CLEAN_HR):
    shutil.rmtree(CLEAN_HR)
if os.path.exists(CLEAN_LR):
    shutil.rmtree(CLEAN_LR)
Path(CLEAN_HR).mkdir(parents=True, exist_ok=True)
Path(CLEAN_LR).mkdir(parents=True, exist_ok=True)

for fn in tqdm(clean_names, desc="Link/copy clean pairs"):
    safe_link_or_copy(hr_files[fn], os.path.join(CLEAN_HR, fn))
    safe_link_or_copy(lr_files[fn], os.path.join(CLEAN_LR, fn))

TRAIN_HR_USED = CLEAN_HR
TRAIN_LR_USED = CLEAN_LR
num_train_images = len(clean_names)
steps_per_epoch = math.ceil(num_train_images / BATCH_SIZE)

with open(CLEAN_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump({
        "clean_psnr_threshold": CLEAN_PSNR_THRESHOLD,
        "num_common": len(common),
        "num_clean": len(clean_names),
        "batch_size": BATCH_SIZE,
        "steps_per_epoch": steps_per_epoch,
    }, f, indent=2)

print("TRAIN_HR_USED:", TRAIN_HR_USED)
print("TRAIN_LR_USED:", TRAIN_LR_USED)
print("num_train_images:", num_train_images)
print("steps_per_epoch:", steps_per_epoch)


In [ ]:
import yaml
import torch

def iter_from_path(path):
    name = Path(str(path)).stem
    nums = re.findall(r"\d+", name)
    return int(nums[-1]) if nums else -1

def find_latest_resume(exp_name):
    state_patterns = [
        f"/kaggle/input/**/{exp_name}/training_states/*.state",
        f"/kaggle/input/**/resume_package/{exp_name}/training_states/*.state",
        "/kaggle/input/**/training_states/*.state",
    ]
    states = []
    for pat in state_patterns:
        states += glob.glob(pat, recursive=True)
    states = [p for p in sorted(set(states)) if os.path.isfile(p)]
    if not states:
        return None, None, 0
    states = sorted(states, key=iter_from_path)
    state_path = states[-1]
    resume_iter = iter_from_path(state_path)
    state_p = Path(state_path)
    candidates = []
    try:
        exp_root = state_p.parents[1]
        candidates += glob.glob(str(exp_root / "models" / f"net_g_{resume_iter}.pth"))
        candidates += glob.glob(str(exp_root / "models" / "net_g_latest.pth"))
    except Exception:
        pass
    candidates += glob.glob(f"/kaggle/input/**/net_g_{resume_iter}.pth", recursive=True)
    candidates += glob.glob(f"/kaggle/input/**/net_g_latest.pth", recursive=True)
    candidates = [p for p in sorted(set(candidates)) if os.path.isfile(p)]
    if not candidates:
        raise FileNotFoundError(f"Found state {state_path}, but cannot find matching net_g_{resume_iter}.pth")
    exact = [p for p in candidates if Path(p).name == f"net_g_{resume_iter}.pth"]
    model_path = exact[0] if exact else candidates[0]
    return model_path, state_path, resume_iter

RESUME_MODEL_INPUT = None
RESUME_STATE_INPUT = None
RESUME_ITER = 0
if AUTO_RESUME_FROM_INPUT:
    RESUME_MODEL_INPUT, RESUME_STATE_INPUT, RESUME_ITER = find_latest_resume(EXP_NAME)

if RESUME_STATE_INPUT is None:
    print("No previous resume state found. This is session 1: 0 -> 10000.")
    RESUME_ITER = 0
    AUTO_RESUME = False
    SESSION_TARGET_ITER = min(ITERS_PER_SESSION, TOTAL_TARGET_ITER)
else:
    print("Found previous resume state:", RESUME_STATE_INPUT)
    print("Found previous model:", RESUME_MODEL_INPUT)
    print("RESUME_ITER:", RESUME_ITER)
    AUTO_RESUME = True
    SESSION_TARGET_ITER = min(RESUME_ITER + ITERS_PER_SESSION, TOTAL_TARGET_ITER)

TOTAL_ITER = int(SESSION_TARGET_ITER)
FINAL_TOTAL_ITER = TOTAL_ITER

if AUTO_SKIP_IF_TARGET_REACHED and RESUME_ITER >= TOTAL_TARGET_ITER:
    RUN_TRAINING = False
    print("Target already reached. RUN_TRAINING=False")

print("This session target TOTAL_ITER:", TOTAL_ITER)

exp_dir = Path(HAT_DIR) / "experiments" / EXP_NAME
model_dir = exp_dir / "models"
state_dir = exp_dir / "training_states"
vis_dir = exp_dir / "visualization"

if FORCE_DELETE_EXISTING_EXPERIMENT and not AUTO_RESUME and exp_dir.exists():
    shutil.rmtree(exp_dir)
    print("Deleted old experiment:", exp_dir)

for d in [exp_dir, model_dir, state_dir, vis_dir]:
    d.mkdir(parents=True, exist_ok=True)

if AUTO_RESUME:
    work_model = model_dir / f"net_g_{RESUME_ITER}.pth"
    work_latest = model_dir / "net_g_latest.pth"
    work_state = state_dir / f"{RESUME_ITER}.state"
    shutil.copy2(RESUME_MODEL_INPUT, work_model)
    shutil.copy2(RESUME_MODEL_INPUT, work_latest)

    state = torch.load(RESUME_STATE_INPUT, map_location="cpu")
    old_epoch = int(state.get("epoch", 0))
    state["iter"] = int(RESUME_ITER)
    total_epochs = math.ceil(int(TOTAL_ITER) / int(steps_per_epoch))
    new_epoch = max(0, min(int(RESUME_ITER // steps_per_epoch), max(0, total_epochs - 1)))
    state["epoch"] = new_epoch
    torch.save(state, work_state)

    MODEL_PATH = str(work_model)
    PRETRAIN_PATH = str(work_model)
    RESUME_STATE_PATH = str(work_state)
    print("Restored resume files to working:")
    print("model:", MODEL_PATH)
    print("state:", RESUME_STATE_PATH)
    print("old_epoch:", old_epoch, "new_epoch:", new_epoch)
else:
    MODEL_PATH = PRETRAIN_PATH
    RESUME_STATE_PATH = None

base_yml = os.path.join(HAT_DIR, "options/train/train_HAT_SRx4_finetune_from_SRx2.yml")
if not os.path.exists(base_yml):
    raise FileNotFoundError(f"Base yaml not found: {base_yml}")
print("Base yaml:", base_yml)

with open(base_yml, "r", encoding="utf-8") as f:
    opt = yaml.safe_load(f)

opt["name"] = EXP_NAME
opt["model_type"] = "HATModel"
opt["scale"] = 4
opt["num_gpu"] = 1
opt["manual_seed"] = SEED
opt["auto_resume"] = False

opt["datasets"] = {}
opt["datasets"]["train"] = {
    "name": "SR in Video Games cleaned",
    "type": "PairedImageDataset",
    "dataroot_gt": TRAIN_HR_USED,
    "dataroot_lq": TRAIN_LR_USED,
    "io_backend": {"type": "disk"},
    "gt_size": int(GT_SIZE),
    "use_hflip": bool(USE_HFLIP),
    "use_rot": bool(USE_ROT),
    "use_shuffle": True,
    "num_worker_per_gpu": 2,
    "batch_size_per_gpu": int(BATCH_SIZE),
    "dataset_enlarge_ratio": 1,
    "prefetch_mode": "cuda",
    "pin_memory": True,
    "phase": "train",
    "scale": 4,
}
opt.pop("val", None)

opt.setdefault("path", {})
opt["path"]["pretrain_network_g"] = MODEL_PATH
opt["path"]["param_key_g"] = PARAM_KEY_G
opt["path"]["strict_load_g"] = bool(STRICT_LOAD_G)
opt["path"]["resume_state"] = RESUME_STATE_PATH if AUTO_RESUME else None

opt.setdefault("train", {})
opt["train"]["total_iter"] = int(TOTAL_ITER)
opt["train"]["warmup_iter"] = int(WARMUP_ITER)
opt["train"]["ema_decay"] = float(EMA_DECAY)
opt["train"]["optim_g"] = {
    "type": "Adam",
    "lr": float(LEARNING_RATE),
    "weight_decay": float(WEIGHT_DECAY),
    "betas": [float(ADAM_BETAS[0]), float(ADAM_BETAS[1])],
}

if LOSS_TYPE.lower() == "l1":
    opt["train"]["pixel_opt"] = {"type": "L1Loss", "loss_weight": 1.0, "reduction": "mean"}
elif LOSS_TYPE.lower() == "charbonnier":
    opt["train"]["pixel_opt"] = {"type": "CharbonnierLoss", "loss_weight": 1.0, "reduction": "mean", "eps": 1e-6}
elif LOSS_TYPE.lower() == "mse":
    opt["train"]["pixel_opt"] = {"type": "MSELoss", "loss_weight": 1.0, "reduction": "mean"}
else:
    raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

if USE_COSINE_LR:
    opt["train"]["scheduler"] = {
        "type": "CosineAnnealingRestartLR",
        "periods": [int(x) for x in COSINE_PERIODS],
        "restart_weights": [float(x) for x in COSINE_RESTART_WEIGHTS],
        "eta_min": float(COSINE_ETA_MIN),
    }
elif USE_CONSTANT_LR:
    opt["train"]["scheduler"] = {"type": "MultiStepRestartLR", "milestones": [10**12], "gamma": 1.0}
else:
    opt["train"]["scheduler"] = {
        "type": "MultiStepRestartLR",
        "milestones": [max(1, int(TOTAL_ITER * 0.75)), max(1, int(TOTAL_ITER * 0.90))],
        "gamma": 0.5,
    }

opt.setdefault("logger", {})
opt["logger"]["print_freq"] = 100
opt["logger"]["save_checkpoint_freq"] = int(max(100, min(SAVE_CHECKPOINT_FREQ, TOTAL_ITER)))
opt["logger"]["use_tb_logger"] = False
opt["pbar"] = True

out_yml = os.path.join(HAT_DIR, f"options/train/{EXP_NAME}.yml")
with open(out_yml, "w", encoding="utf-8") as f:
    yaml.safe_dump(opt, f, sort_keys=False)

summary = {
    "name": opt["name"],
    "session_resume_iter": RESUME_ITER,
    "session_target_iter": TOTAL_ITER,
    "total_target_iter": TOTAL_TARGET_ITER,
    "auto_resume": AUTO_RESUME,
    "resume_state": opt["path"]["resume_state"],
    "pretrain_network_g": opt["path"]["pretrain_network_g"],
    "model_variant": MODEL_VARIANT,
    "batch_size": opt["datasets"]["train"]["batch_size_per_gpu"],
    "gt_size": opt["datasets"]["train"]["gt_size"],
    "loss": opt["train"]["pixel_opt"],
    "learning_rate_start": opt["train"]["optim_g"]["lr"],
    "scheduler": opt["train"]["scheduler"],
    "save_checkpoint_freq": opt["logger"]["save_checkpoint_freq"],
}
print("Wrote:", out_yml)
print(yaml.safe_dump(summary, sort_keys=False))


In [ ]:
if RUN_TRAINING and TOTAL_ITER > 0:
    run(f"python hat/train.py -opt {out_yml}", cwd=HAT_DIR)
else:
    print("RUN_TRAINING=False or TOTAL_ITER=0, skipping training.")

exp_dir = Path(HAT_DIR) / "experiments" / EXP_NAME
model_dir = exp_dir / "models"
state_dir = exp_dir / "training_states"

latest_pth = model_dir / "net_g_latest.pth"
effective_pth = model_dir / f"net_g_{TOTAL_ITER}.pth"

if not latest_pth.exists():
    numbered = glob.glob(str(model_dir / "net_g_*.pth"))
    def iter_key(p):
        nums = re.findall(r"\d+", Path(p).stem)
        return int(nums[-1]) if nums else -1
    numbered = sorted(numbered, key=iter_key)
    if numbered:
        latest_pth = Path(numbered[-1])

if latest_pth.exists():
    shutil.copy2(latest_pth, effective_pth)
    shutil.copy2(latest_pth, Path(WORK_DIR) / f"net_g_{TOTAL_ITER}.pth")
    print("Exported checkpoint:", effective_pth)
    print("Exported checkpoint:", Path(WORK_DIR) / f"net_g_{TOTAL_ITER}.pth")
else:
    print("No latest checkpoint found:", latest_pth)

def safe_copy2(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        print("Skip missing file:", src)
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if dst.exists() and os.path.samefile(src, dst):
            print("Skip same file:", src)
            return
    except FileNotFoundError:
        pass
    shutil.copy2(src, dst)

if EXPORT_RESUME_PACKAGE:
    resume_dir = Path(WORK_DIR) / "resume_package" / EXP_NAME
    if exp_dir.exists():
        resume_dir.mkdir(parents=True, exist_ok=True)
        for sub in ["models", "training_states"]:
            src_sub = exp_dir / sub
            dst_sub = resume_dir / sub
            if src_sub.exists():
                shutil.copytree(src_sub, dst_sub, dirs_exist_ok=True)
        if os.path.exists(out_yml):
            safe_copy2(out_yml, resume_dir / "train_config.yml")
        if os.path.exists(REPORT_CSV):
            safe_copy2(REPORT_CSV, resume_dir / "clean_pair_report.csv")
        if os.path.exists(CLEAN_CONFIG_JSON):
            safe_copy2(CLEAN_CONFIG_JSON, resume_dir / "clean_config.json")
        print("Exported resume package:", resume_dir)
        print("Model files:")
        for p in sorted(glob.glob(str(resume_dir / "models" / "*.pth"))):
            print(" -", p)
        print("State files:")
        for p in sorted(glob.glob(str(resume_dir / "training_states" / "*.state"))):
            print(" -", p)
    else:
        print("No experiment directory found:", exp_dir)


In [ ]:
if not RUN_INFERENCE_AFTER_TRAIN:
    print("RUN_INFERENCE_AFTER_TRAIN=False, skipping submission generation.")
else:
    from PIL import Image
    import torch
    import torch.nn.functional as F

    os.chdir(HAT_DIR)
    if HAT_DIR not in sys.path:
        sys.path.insert(0, HAT_DIR)

    try:
        import hat.archs.hat_arch  # noqa
    except Exception as e:
        print("Import warning:", repr(e))

    from basicsr.archs import build_network

    with open(out_yml, "r", encoding="utf-8") as f:
        infer_opt = yaml.safe_load(f)

    model = build_network(infer_opt["network_g"])
    model = model.float().eval().cuda()

    def model_iter_key(path):
        name = Path(path).stem.lower()
        if "latest" in name:
            return 10**12
        nums = re.findall(r"\d+", name)
        return int(nums[-1]) if nums else -1

    ckpt_dir = Path(HAT_DIR) / "experiments" / EXP_NAME / "models"
    ckpt_candidates = glob.glob(str(ckpt_dir / "net_g*.pth"))
    if not ckpt_candidates:
        raise FileNotFoundError(f"No net_g*.pth found in: {ckpt_dir}")

    latest_candidates = [p for p in ckpt_candidates if Path(p).name == "net_g_latest.pth"]
    latest_ckpt = latest_candidates[0] if latest_candidates else sorted(
        ckpt_candidates,
        key=lambda p: (model_iter_key(p), os.path.getmtime(p))
    )[-1]

    numbered = [p for p in ckpt_candidates if model_iter_key(p) < 10**12]
    current_iter = max([model_iter_key(p) for p in numbered], default=TOTAL_ITER)

    loadnet = torch.load(latest_ckpt, map_location="cpu")
    if isinstance(loadnet, dict) and "params_ema" in loadnet:
        state = loadnet["params_ema"]
        load_key = "params_ema"
    elif isinstance(loadnet, dict) and "params" in loadnet:
        state = loadnet["params"]
        load_key = "params"
    else:
        state = loadnet
        load_key = "raw"

    model.load_state_dict(state, strict=True)
    model = model.float().eval().cuda()

    print("Loaded checkpoint:", latest_ckpt)
    print("Load key:", load_key)
    print("Current iter:", current_iter)

    def encode(img: np.ndarray) -> str:
        img_to_encode = img.astype(np.uint8).flatten()
        img_to_encode = np.append(img_to_encode, -1)

        cnt, rle = 1, []
        for i in range(1, img_to_encode.shape[0]):
            if img_to_encode[i] == img_to_encode[i - 1]:
                cnt += 1
                if cnt > 255:
                    rle += [int(img_to_encode[i - 1]), 255]
                    cnt = 1
            else:
                rle += [int(img_to_encode[i - 1]), cnt]
                cnt = 1

        compressed = zlib.compress(bytes(rle), zlib.Z_BEST_COMPRESSION)
        return str(base64.b64encode(compressed))
    ops_for_mode = {
        0: [],
        1: ["hflip"],
        2: ["vflip"],
        3: ["hflip", "vflip"],
        4: ["transpose"],
        5: ["transpose", "hflip"],
        6: ["transpose", "vflip"],
        7: ["transpose", "hflip", "vflip"],
    }

    def apply_ops(x, ops):
        for op in ops:
            if op == "hflip":
                x = torch.flip(x, dims=[-1])
            elif op == "vflip":
                x = torch.flip(x, dims=[-2])
            elif op == "transpose":
                x = x.transpose(-1, -2)
        return x

    def invert_ops(x, ops):
        for op in reversed(ops):
            if op == "hflip":
                x = torch.flip(x, dims=[-1])
            elif op == "vflip":
                x = torch.flip(x, dims=[-2])
            elif op == "transpose":
                x = x.transpose(-1, -2)
        return x

    @torch.no_grad()
    def forward_once(x):
        x = x.float()
        _, _, h, w = x.shape

        pad_h = (WINDOW_SIZE - h % WINDOW_SIZE) % WINDOW_SIZE
        pad_w = (WINDOW_SIZE - w % WINDOW_SIZE) % WINDOW_SIZE

        if pad_h or pad_w:
            x_pad = F.pad(x, (0, pad_w, 0, pad_h), mode="reflect")
        else:
            x_pad = x

        y = model(x_pad.float())
        return y[:, :, :h * 4, :w * 4].float()

    @torch.no_grad()
    def predict_tensor(x):
        x = x.float()

        if not RUN_TTA_SUBMISSION:
            return forward_once(x)

        outs = []
        for m in TTA_MODES:
            ops = ops_for_mode[m]
            xa = apply_ops(x, ops)
            ya = forward_once(xa)
            outs.append(invert_ops(ya, ops))

        return torch.stack(outs, dim=0).mean(dim=0)

    sub = pd.read_csv(SAMPLE_SUB)[["id", "filename", "rle"]].copy()
    progress_csv = "/kaggle/working/submission_progress.csv"

    print("Predicting", len(sub), "test images.")
    print("TTA enabled:", RUN_TTA_SUBMISSION)
    print("TTA modes:", TTA_MODES if RUN_TTA_SUBMISSION else [0])

    for i, fn in enumerate(tqdm(sub["filename"].tolist(), desc="Predicting test images")):
        lr_path = os.path.join(TEST_LR, fn)
        img = Image.open(lr_path).convert("RGB")
        arr = np.array(img).astype(np.float32) / 255.0

        x = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).cuda().float()
        y = predict_tensor(x)

        out_rgb = y.squeeze(0).permute(1, 2, 0).clamp(0, 1).cpu().numpy()
        out_rgb = np.rint(out_rgb * 255.0).astype(np.uint8)

        out_to_encode = out_rgb[..., ::-1] if ENCODE_BGR else out_rgb
        sub.at[i, "rle"] = encode(out_to_encode)

        if (i + 1) % 100 == 0:
            sub.to_csv(progress_csv, index=False)

    suffix = "tta" if RUN_TTA_SUBMISSION else "fast"
    out_iter_csv = f"/kaggle/working/submission_iter_{current_iter}_{suffix}.csv"
    out_csv = "/kaggle/working/submission.csv"

    sub.to_csv(out_iter_csv, index=False)
    sub.to_csv(out_csv, index=False)

    best_export = "/kaggle/working/best_model_latest.pth"
    shutil.copy2(latest_ckpt, best_export)

    print("Saved:", out_iter_csv)
    print("Saved:", out_csv)
    print("Exported latest model:", best_export)

    print(sub.head())
    print("Shape:", sub.shape)
    print("Missing rle:", sub["rle"].isna().sum())
    print("Starts with b':", sub["rle"].astype(str).str.startswith("b'").mean())
    print("Example rle:", sub.loc[0, "rle"])
